#### Installations des packages nécessaires

In [ ]:
# Installations des librairies nécessaires
#pip install pandas numpy matplotlib seaborn statsmodels pmdarima scikit-learn missingno openpyxl 

# en cas d'erreur lors de l'installation de prophet, vous pouvez installer les librairies suivantes avant d'installer prophet
#pip install pystan==2.19.1.1
#pip install prophet 


SyntaxError: invalid syntax (3653075892.py, line 2)

#### 0- Importation des packages 

In [16]:
# ==========================================================
# SYSTÈME COMPLET DE SURVEILLANCE MPOX - SÉNÉGAL
# CIBLE : Nb_Cas (Nombre de cas par semaine)
# ==========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import seasonal_decompose
from pmdarima import auto_arima
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression, RFE, mutual_info_regression
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
import warnings
warnings.filterwarnings('ignore')
from datetime import datetime, timedelta
import os
import glob



#### 1- La fonction de chargement de la base de donnée

In [17]:
# ============================================================
# 1. CHARGEMENT
# ============================================================

def trouver_fichier_mpox():
    """Recherche automatiquement le fichier Excel"""
    print("\n" + "="*70)
    print("RECHERCHE DU FICHIER DE DONNÉES")
    print("="*70)
    
    noms = [
        'Base_mpox_sen_S_05_2026.xlsx',  # notre base de données principale
        'Base_mpox_sen_5.05_2026.csv',
        'Base_mpox_sen*.xlsx',
        '*mpox*.xlsx',
        'Base_mpox*.xlsx'
    ]
    
    print("\n🔍 Recherche en cours...")
    
    for nom in noms:
        fichiers = glob.glob(nom)
        for fichier in fichiers:
            if os.path.exists(fichier):
                print(f"✅ Fichier trouvé: {fichier}")
                return fichier
    
    for root, dirs, files in os.walk('.'):
        for file in files:
            if 'mpox' in file.lower() and (file.endswith('.xlsx') or file.endswith('.xls')):
                chemin = os.path.join(root, file)
                print(f"✅ Fichier trouvé: {chemin}")
                return chemin
    
    print("❌ Aucun fichier trouvé")
    return None

def charger_donnees(fichier=None):
    """Charge la base de données"""
    print("\n" + "="*70)
    print("CHARGEMENT DE LA BASE DE DONNÉES")
    print("="*70)
    
    if fichier is None:
        fichier = trouver_fichier_mpox()
        if fichier is None:
            return None
    
    try:
        if fichier.endswith('.csv'):
            base = pd.read_csv(fichier, encoding='utf-8')
        else:
            base = pd.read_excel(fichier)
        
        print(f"\n✅ Base chargée avec succès")
        print(f"   - {base.shape[0]} lignes")
        print(f"   - {base.shape[1]} colonnes")
        return base
    except Exception as e:
        print(f"❌ Erreur: {e}")
        return None

#### 2- La Fonction de creation de la variable cible (Nb_Cas)

In [18]:

# ============================================================
# 2. CRÉATION DE LA VARIABLE CIBLE Nb_Cas
# ============================================================

def creer_variable_cible(df):
    """
    Crée la variable cible principale Nb_Cas (nombre de cas par semaine)
    """
    print("\n" + "="*70)
    print("🎯 CRÉATION DE LA VARIABLE CIBLE : Nb_Cas")
    print("="*70)
    
    print("""
    📚 OBJECTIF DE LA VARIABLE CIBLE :
    ----------------------------------
    La variable Nb_Cas représente le nombre de cas suspects recensés
    chaque semaine épidémiologique. C'est la variable que nous cherchons
    à prédire pour anticiper l'évolution de la maladie.
    
    📊 JUSTIFICATION :
    -----------------
    1. Adaptée aux modèles de séries temporelles (ARIMA, SARIMA)
    2. Permet des prévisions à court terme
    3. Utile pour les indicateurs d'alerte
    4. Correspond à l'objectif de surveillance épidémiologique
    """)
    
    # Vérifier la présence de Semaine_Epi
    if 'Semaine_Epi' not in df.columns:
        print("❌ Colonne 'Semaine_Epi' non trouvée")
        return df
    
    # Créer la variable cible : nombre de cas par semaine
    df_hebdo = df.groupby('Semaine_Epi').size().reset_index(name='Nb_Cas')
    
    # Fusionner avec le DataFrame original
    df = df.merge(df_hebdo, on='Semaine_Epi', how='left')
    
    print(f"\n✅ Variable cible 'Nb_Cas' créée avec succès")
    print(f"\n📊 STATISTIQUES DE LA CIBLE :")
    print("-"*50)
    print(f"   Total des cas           : {df['Nb_Cas'].sum()}")
    print(f"   Nombre de semaines      : {df['Semaine_Epi'].nunique()}")
    print(f"   Moyenne par semaine     : {df['Nb_Cas'].mean():.2f}")
    print(f"   Médiane par semaine     : {df['Nb_Cas'].median():.0f}")
    print(f"   Écart-type              : {df['Nb_Cas'].std():.2f}")
    print(f"   Minimum                 : {df['Nb_Cas'].min()}")
    print(f"   Maximum                 : {df['Nb_Cas'].max()}")
    
    # Distribution
    print(f"\n📊 DISTRIBUTION DE Nb_Cas :")
    print("-"*50)
    dist = df.groupby('Semaine_Epi')['Nb_Cas'].first().value_counts().sort_index()
    for val, count in dist.head(10).items():
        pct = count / len(dist) * 100
        bar = '█' * int(pct / 2)
        print(f"   {val:>3} cas : {count:>3} semaines ({pct:>5.1f}%) {bar}")
    
    return df


#### 3- La fonction d'analyse détaillé des NA

In [19]:

# ============================================================
# 3. ANALYSE DÉTAILLÉE DES NA
# ============================================================

def analyse_approfondie_na(df):
    """Analyse approfondie des valeurs manquantes"""
    print("\n" + "="*70)
    print("📊 ANALYSE APPROFONDIE DES VALEURS MANQUANTES")
    print("="*70)
    
    print("""
    📚 CAUSES POSSIBLES DES VALEURS MANQUANTES :
    --------------------------------------------
    1. Données non collectées : Variable non renseignée systématiquement
    2. Erreurs de saisie : Oubli lors de la collecte
    3. Non applicabilité : Variable non concernée pour certains patients
    4. Données non disponibles : Résultats en attente
    
    📊 STRATÉGIES DE TRAITEMENT :
    -----------------------------
    1. Suppression : Si > 80% de NA
    2. Imputation simple : Médiane (numérique) / Mode (catégoriel) si < 20%
    3. Imputation avancée : KNN si 20-80%
    """)
    
    prop_manquantes = df.isnull().mean() * 100
    prop_manquantes = prop_manquantes.sort_values(ascending=False)
    prop_manquantes_nonzero = prop_manquantes[prop_manquantes > 0]
    
    a_supprimer = prop_manquantes[prop_manquantes > 80].index.tolist()
    a_imputer = prop_manquantes[(prop_manquantes > 20) & (prop_manquantes <= 80)].index.tolist()
    a_imputer_simple = prop_manquantes[(prop_manquantes <= 20) & (prop_manquantes > 0)].index.tolist()
    
    print("\n📊 RÉPARTITION DES NA PAR SEUIL :")
    print("-"*60)
    for seuil_bas, seuil_haut in [(0,5), (5,20), (20,50), (50,80), (80,100)]:
        count = sum((prop_manquantes_nonzero > seuil_bas) & (prop_manquantes_nonzero <= seuil_haut))
        if count > 0:
            print(f"   {seuil_bas:>3}% - {seuil_haut:>3}% : {count:>3} variables")
    
    print(f"\n📊 VARIABLES À SUPPRIMER (> 80% NA) : {len(a_supprimer)}")
    print("-"*60)
    if a_supprimer:
        for var in a_supprimer:
            print(f"   ❌ {var:<45} : {prop_manquantes[var]:>5.1f}% NA")
    else:
        print("   Aucune")
    
    print(f"\n📊 VARIABLES À IMPUTER PAR KNN (20-80% NA) : {len(a_imputer)}")
    print("-"*60)
    if a_imputer:
        for var in a_imputer:
            print(f"   ⚠️ {var:<45} : {prop_manquantes[var]:>5.1f}% NA")
    else:
        print("   Aucune")
    
    print(f"\n📊 VARIABLES À IMPUTER SIMPLE (< 20% NA) : {len(a_imputer_simple)}")
    print("-"*60)
    if a_imputer_simple:
        for var in a_imputer_simple:
            print(f"   ℹ️ {var:<45} : {prop_manquantes[var]:>5.1f}% NA")
    else:
        print("   Aucune")
    
    print(f"\n📊 RÉSUMÉ GLOBAL :")
    print("-"*60)
    print(f"   Total cellules         : {df.shape[0] * df.shape[1]:>10,}")
    print(f"   Total NA               : {df.isnull().sum().sum():>10,}")
    print(f"   Pourcentage global     : {df.isnull().sum().sum()/(df.shape[0]*df.shape[1])*100:>9.1f}%")
    print(f"   Variables à supprimer  : {len(a_supprimer):>10}")
    print(f"   Variables à imputer    : {len(a_imputer) + len(a_imputer_simple):>10}")
    print(f"   Variables conservées   : {df.shape[1] - len(a_supprimer):>10}")
    
    return {
        'a_supprimer': a_supprimer,
        'a_imputer': a_imputer,
        'a_imputer_simple': a_imputer_simple,
        'prop_manquantes': prop_manquantes
    }


#### 4- La fonction de traitement détaillé des NA avec les 03 Stratégies

In [20]:
# ============================================================
# 4. TRAITEMENT DÉTAILLÉ DES NA
# ============================================================

def traiter_na_avance(df, config_na):
    """Traitement avancé avec affichage détaillé"""
    print("\n" + "="*70)
    print("🔧 TRAITEMENT DES VALEURS MANQUANTES")
    print("="*70)
    
    df_clean = df.copy()
    nb_supprimees = 0
    nb_imputees_simple = 0
    nb_imputees_knn = 0
    vars_imputees_simple = {}
    vars_imputees_knn = {}
    
    # ---- ÉTAPE 1: SUPPRESSION ----
    print("\n🔧 ÉTAPE 1: SUPPRESSION DES VARIABLES (> 80% NA)")
    print("-"*60)
    vars_a_supprimer = config_na['a_supprimer']
    if vars_a_supprimer:
        print(f"Suppression de {len(vars_a_supprimer)} variables:\n")
        print(f"   {'#':<4} {'Variable':<45} {'% NA':>8} {'NA':>8}")
        print(f"   {'-'*4} {'-'*45} {'-'*8} {'-'*8}")
        for i, var in enumerate(vars_a_supprimer, 1):
            pct = config_na['prop_manquantes'][var]
            nb_na = df[var].isnull().sum()
            print(f"   {i:<4} {var:<45} {pct:>7.1f}% {nb_na:>8}")
        df_clean = df_clean.drop(columns=vars_a_supprimer, errors='ignore')
        nb_supprimees = len(vars_a_supprimer)
    else:
        print("   Aucune variable à supprimer")
    
    # ---- ÉTAPE 2: IMPUTATION SIMPLE ----
    print("\n🔧 ÉTAPE 2: IMPUTATION SIMPLE (< 20% NA)")
    print("-"*60)
    
    var_num = df_clean.select_dtypes(include=[np.number]).columns
    var_cat = df_clean.select_dtypes(include=['object']).columns
    vars_simple = config_na['a_imputer_simple']
    
    if vars_simple:
        print(f"\n   Variables numériques (médiane):")
        print(f"   {'#':<4} {'Variable':<40} {'Médiane':>12} {'NA':>6}")
        print(f"   {'-'*4} {'-'*40} {'-'*12} {'-'*6}")
        i = 0
        for var in vars_simple:
            if var in var_num and df_clean[var].isnull().sum() > 0:
                i += 1
                mediane = df_clean[var].median()
                if pd.isna(mediane):
                    mediane = df_clean[var].mean()
                if not pd.isna(mediane):
                    nb_na = df_clean[var].isnull().sum()
                    df_clean[var] = df_clean[var].fillna(mediane)
                    vars_imputees_simple[var] = {'methode': 'médiane', 'valeur': mediane, 'nb_na': nb_na}
                    print(f"   {i:<4} {var:<40} {mediane:>12.2f} {nb_na:>6}")
        if i == 0:
            print("   (aucune)")
        
        print(f"\n   Variables catégorielles (mode):")
        print(f"   {'#':<4} {'Variable':<40} {'Mode':>15} {'NA':>6}")
        print(f"   {'-'*4} {'-'*40} {'-'*15} {'-'*6}")
        j = 0
        for var in vars_simple:
            if var in var_cat and df_clean[var].isnull().sum() > 0:
                j += 1
                mode = df_clean[var].mode()[0] if not df_clean[var].mode().empty else 'Non spécifié'
                nb_na = df_clean[var].isnull().sum()
                df_clean[var] = df_clean[var].fillna(mode)
                vars_imputees_simple[var] = {'methode': 'mode', 'valeur': mode, 'nb_na': nb_na}
                print(f"   {j:<4} {var:<40} {str(mode)[:15]:>15} {nb_na:>6}")
        if j == 0:
            print("   (aucune)")
        
        nb_imputees_simple = len(vars_imputees_simple)
        print(f"\n   ✅ {nb_imputees_simple} variables imputées (simple)")
    else:
        print("   Aucune")
    
    # ---- ÉTAPE 3: IMPUTATION KNN ----
    print("\n🔧 ÉTAPE 3: IMPUTATION KNN (20-80% NA)")
    print("-"*60)
    
    vars_knn = [v for v in config_na['a_imputer'] if v in df_clean.columns and v in var_num]
    
    if vars_knn:
        print(f"\n   {len(vars_knn)} variables à imputer par KNN:")
        print(f"   {'#':<4} {'Variable':<45} {'% NA':>8} {'NA':>6}")
        print(f"   {'-'*4} {'-'*45} {'-'*8} {'-'*6}")
        for i, var in enumerate(vars_knn, 1):
            pct = config_na['prop_manquantes'][var]
            nb_na = df_clean[var].isnull().sum()
            print(f"   {i:<4} {var:<45} {pct:>7.1f}% {nb_na:>6}")
        
        try:
            df_knn = df_clean[vars_knn].copy()
            scaler = StandardScaler()
            df_knn_scaled = pd.DataFrame(
                scaler.fit_transform(df_knn), columns=vars_knn, index=df_knn.index
            )
            
            knn_imputer = KNNImputer(n_neighbors=5)
            df_knn_imputed = pd.DataFrame(
                knn_imputer.fit_transform(df_knn_scaled), columns=vars_knn, index=df_knn.index
            )
            
            df_knn_imputed = pd.DataFrame(
                scaler.inverse_transform(df_knn_imputed), columns=vars_knn, index=df_knn.index
            )
            
            for var in vars_knn:
                nb_na = df_clean[var].isnull().sum()
                df_clean[var] = df_knn_imputed[var]
                vars_imputees_knn[var] = {'methode': 'KNN', 'nb_na': nb_na}
            
            nb_imputees_knn = len(vars_knn)
            print(f"\n   ✅ {nb_imputees_knn} variables imputées par KNN (k=5)")
        except Exception as e:
            print(f"   ⚠️ Erreur KNN: {e}, utilisation médiane")
            for var in vars_knn:
                nb_na = df_clean[var].isnull().sum()
                mediane = df_clean[var].median()
                if not pd.isna(mediane):
                    df_clean[var] = df_clean[var].fillna(mediane)
                    vars_imputees_knn[var] = {'methode': 'médiane', 'nb_na': nb_na}
            nb_imputees_knn = len(vars_knn)
    else:
        print("   Aucune")
    
    # ---- RÉSUMÉ ----
    print("\n" + "="*70)
    print("📊 RÉSUMÉ DU TRAITEMENT")
    print("="*70)
    print(f"   Variables supprimées        : {nb_supprimees}")
    print(f"   Variables imputées (simple) : {nb_imputees_simple}")
    print(f"   Variables imputées (KNN)    : {nb_imputees_knn}")
    print(f"   Total variables traitées    : {nb_supprimees + nb_imputees_simple + nb_imputees_knn}")
    print(f"   Shape final                 : {df_clean.shape[0]} × {df_clean.shape[1]}")
    print(f"   NA restants                 : {df_clean.isnull().sum().sum()}")
    
    return df_clean, {
        'nb_supprimees': nb_supprimees,
        'nb_imputees_simple': nb_imputees_simple,
        'nb_imputees_knn': nb_imputees_knn,
        'vars_imputees_simple': vars_imputees_simple,
        'vars_imputees_knn': vars_imputees_knn,
        'vars_supprimees': vars_a_supprimer
    }


#### 5- La fonction de préparation des données

In [21]:
# ============================================================
# 5. PRÉPARATION DES DONNÉES
# ============================================================

def preparer_donnees_complet(base):
    """
    Prépare les données avec features temporelles
    VERSION CORRIGÉE: Nb_Cas agrégée par Date (pas Semaine_Epi)
    """
    print("\n" + "="*70)
    print("PRÉPARATION DES DONNÉES")
    print("="*70)
    
    df = base.copy()
    df.columns = df.columns.str.strip()
    
    # 1. Créer Date à partir de Semaine_Epi
    if 'Semaine_Epi' in df.columns:
        print("\n🔧 Traitement des dates")
        try:
            df['Semaine_Num'] = df['Semaine_Epi'].str.extract(r'(\d+)').astype(float)
            df['Annee_Num'] = df['Semaine_Epi'].str.extract(r'(\d{4})').astype(float)
            
            def semaine_to_date(row):
                try:
                    if pd.notna(row['Annee_Num']) and pd.notna(row['Semaine_Num']):
                        annee = int(row['Annee_Num'])
                        semaine = int(row['Semaine_Num'])
                        debut = datetime(annee, 1, 1)
                        return debut + timedelta(weeks=semaine)
                    return pd.NaT
                except:
                    return pd.NaT
            
            df['Date'] = df.apply(semaine_to_date, axis=1)
            # À ajouter juste après la création de df['Date']
            if 'Date' in df.columns:
                print(f"\n🔍 DIAGNOSTIC DATE:")
                print(f"   Type: {df['Date'].dtype}")
                print(f"   Non-NaT: {df['Date'].notna().sum()}")
                print(f"   Exemples:")
                print(df['Date'].dropna().head(5).to_string())
            print(f"   ✅ Dates créées pour {df['Date'].notna().sum()} lignes")
        except Exception as e:
            print(f"   ⚠️ Erreur: {e}")
    
    # 2. CORRECTION: Créer Nb_Cas À PARTIR DE DATE (pas Semaine_Epi)
    if 'Date' in df.columns:
        print("\n🎯 Création de Nb_Cas (par Date unique)")
        try:
            # Compter les cas par DATE unique
            df_hebdo = df.groupby('Date').size().reset_index(name='Nb_Cas')
            df = df.merge(df_hebdo, on='Date', how='left')
            
            print(f"   ✅ Nb_Cas créée")
            print(f"   - Dates uniques: {df['Date'].nunique()}")
            print(f"   - Total cas: {df['Nb_Cas'].sum()}")
            print(f"   - Moyenne: {df.groupby('Date')['Nb_Cas'].first().mean():.2f}")
            print(f"   - Min/Max: {df['Nb_Cas'].min()}/{df['Nb_Cas'].max()}")
        except Exception as e:
            print(f"   ❌ Erreur création Nb_Cas: {e}")
    
    # 3. Features temporelles
    if 'Date' in df.columns and df['Date'].notna().sum() > 0:
        df['Mois'] = df['Date'].dt.month
        df['Semaine_Annee'] = df['Date'].dt.isocalendar().week
        df['Trimestre'] = df['Date'].dt.quarter
        df['Mois_sin'] = np.sin(2 * np.pi * df['Mois'] / 12)
        df['Mois_cos'] = np.cos(2 * np.pi * df['Mois'] / 12)
        print(f"   ✅ Features temporelles créées")
    
    print(f"\n✅ Données préparées: {df.shape[0]} lignes, {df.shape[1]} colonnes")
    
    # Vérification
    if 'Nb_Cas' not in df.columns:
        print("\n❌ ATTENTION: Nb_Cas non créée !")
        print("   Vérifiez la colonne 'Date'")
    
    return df

#### 6- La fonction de selection de variables avec différents modèles

In [22]:
# ============================================================
# 6. SÉLECTION DE VARIABLES
# ============================================================

def selection_variables_avance(df, target='Nb_Cas', n_features=10):
    """Sélection de variables avec 6 méthodes"""
    print("\n" + "="*70)
    print(f"📊 SÉLECTION DE VARIABLES (CIBLE = {target})")
    print("="*70)
    
    var_num = df.select_dtypes(include=[np.number]).columns.tolist()
    
    if target not in var_num:
        print(f"❌ Cible '{target}' non numérique")
        return None
    
    var_num.remove(target)
    
    # Exclure les variables non pertinentes
    exclure = ['Num_IPD', 'Semaine_Num', 'Annee_Num']
    var_num = [v for v in var_num if v not in exclure]
    
    var_utiles = [v for v in var_num if df[v].notna().sum() > 10 and df[v].nunique() > 3]
    
    if len(var_utiles) < 2:
        print("⚠️ Pas assez de variables")
        return None
    
    print(f"\n📊 Variables candidates : {len(var_utiles)}")
    print(f"   Cible                : {target}")
    print(f"   Observations         : {len(df)}")
    
    df_num = df[var_utiles + [target]].fillna(df[var_utiles + [target]].median())
    X = df_num[var_utiles]
    y = df_num[target]
    
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
    
    results = {}
    all_selected = []
    
    # MÉTHODE 1: CORRÉLATION
    print("\n📊 MÉTHODE 1: CORRÉLATION AVEC LA CIBLE")
    print("-"*60)
    correlations = X.corrwith(y).abs().sort_values(ascending=False)
    print(f"   {'#':<4} {'Variable':<40} {'Corrélation':>12}")
    print(f"   {'-'*4} {'-'*40} {'-'*12}")
    for i, (var, corr) in enumerate(correlations.head(10).items(), 1):
        print(f"   {i:<4} {var:<40} {corr:>12.3f}")
    results['correlation'] = correlations.head(n_features).index.tolist()
    all_selected.extend(results['correlation'])
    
    # MÉTHODE 2: SELECTKBEST
    print("\n📊 MÉTHODE 2: SELECTKBEST (F-TEST)")
    print("-"*60)
    try:
        selector_kbest = SelectKBest(score_func=f_regression, k=min(n_features, len(var_utiles)))
        selector_kbest.fit(X_scaled, y)
        results['kbest'] = X.columns[selector_kbest.get_support()].tolist()
        print(f"   {'#':<4} {'Variable':<40} {'Score F':>12}")
        print(f"   {'-'*4} {'-'*40} {'-'*12}")
        for i, var in enumerate(results['kbest'], 1):
            score = selector_kbest.scores_[X.columns.get_loc(var)]
            print(f"   {i:<4} {var:<40} {score:>12.2f}")
        all_selected.extend(results['kbest'])
    except Exception as e:
        print(f"   ⚠️ Erreur: {e}")
        results['kbest'] = []
    
    # MÉTHODE 3: RFE
    print("\n📊 MÉTHODE 3: RFE")
    print("-"*60)
    try:
        rfe = RFE(estimator=LinearRegression(), n_features_to_select=min(n_features, len(var_utiles)))
        rfe.fit(X_scaled, y)
        results['rfe'] = X.columns[rfe.get_support()].tolist()
        print(f"   {'#':<4} {'Variable':<40} {'Rang':>8}")
        print(f"   {'-'*4} {'-'*40} {'-'*8}")
        for i, var in enumerate(results['rfe'], 1):
            ranking = rfe.ranking_[X.columns.get_loc(var)]
            print(f"   {i:<4} {var:<40} {ranking:>8}")
        all_selected.extend(results['rfe'])
    except Exception as e:
        print(f"   ⚠️ Erreur: {e}")
        results['rfe'] = []
    
    # MÉTHODE 4: MUTUAL INFORMATION
    print("\n📊 MÉTHODE 4: INFORMATION MUTUELLE")
    print("-"*60)
    try:
        mi_scores = mutual_info_regression(X_scaled, y)
        mi_series = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)
        results['mi'] = mi_series.head(n_features).index.tolist()
        print(f"   {'#':<4} {'Variable':<40} {'Info Mutuelle':>15}")
        print(f"   {'-'*4} {'-'*40} {'-'*15}")
        for i, (var, score) in enumerate(mi_series.head(10).items(), 1):
            print(f"   {i:<4} {var:<40} {score:>15.3f}")
        all_selected.extend(results['mi'])
    except Exception as e:
        print(f"   ⚠️ Erreur: {e}")
        results['mi'] = []
    
    # MÉTHODE 5: LASSO
    print("\n📊 MÉTHODE 5: LASSO")
    print("-"*60)
    try:
        lasso = Lasso(alpha=0.01)
        lasso.fit(X_scaled, y)
        lasso_coef = pd.Series(lasso.coef_, index=X.columns).abs()
        results['lasso'] = lasso_coef[lasso_coef > 0.001].head(n_features).index.tolist()
        print(f"   {'#':<4} {'Variable':<40} {'Coefficient':>15}")
        print(f"   {'-'*4} {'-'*40} {'-'*15}")
        for i, var in enumerate(results['lasso'], 1):
            coef = lasso.coef_[X.columns.get_loc(var)]
            print(f"   {i:<4} {var:<40} {coef:>15.3f}")
        all_selected.extend(results['lasso'])
    except Exception as e:
        print(f"   ⚠️ Erreur: {e}")
        results['lasso'] = []
    
    # MÉTHODE 6: RANDOM FOREST
    print("\n📊 MÉTHODE 6: RANDOM FOREST")
    print("-"*60)
    try:
        rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        rf.fit(X_scaled, y)
        rf_importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
        results['rf'] = rf_importance.head(n_features).index.tolist()
        print(f"   {'#':<4} {'Variable':<40} {'Importance':>15}")
        print(f"   {'-'*4} {'-'*40} {'-'*15}")
        for i, (var, imp) in enumerate(rf_importance.head(10).items(), 1):
            print(f"   {i:<4} {var:<40} {imp:>15.3f}")
        all_selected.extend(results['rf'])
    except Exception as e:
        print(f"   ⚠️ Erreur: {e}")
        results['rf'] = []
    
    # SYNTHÈSE
    print("\n" + "="*70)
    print("📊 SYNTHÈSE DE LA SÉLECTION")
    print("="*70)
    
    if all_selected:
        vote_count = pd.Series(all_selected).value_counts()
        
        print(f"\n   {'#':<4} {'Variable':<40} {'Votes':>8} {'%':>8}")
        print(f"   {'-'*4} {'-'*40} {'-'*8} {'-'*8}")
        for i, (var, count) in enumerate(vote_count.head(15).items(), 1):
            pct = count / 6 * 100
            print(f"   {i:<4} {var:<40} {count:>7}/6 {pct:>7.0f}%")
        
        final_features = vote_count[vote_count >= 2].index.tolist()
        if not final_features:
            final_features = vote_count.head(n_features).index.tolist()
        
        print(f"\n✅ Variables retenues ({len(final_features)}) :")
        for i, var in enumerate(final_features, 1):
            votes = vote_count[var]
            print(f"   {i:>2}. {var:<40} ({votes}/6)")
        
        return {
            'final_features': final_features,
            'votes': vote_count,
            'results': results,
            'X': X,
            'y': y
        }
    
    return None


#### 7- La fonction de choix des modèles de séries temporelles

In [23]:
# ============================================================
# 7. MODÈLES DE SÉRIES TEMPORELLES
# ============================================================

# Modèles ARIMA avec auto_arima pour la sélection des paramètres
def modele_arima(series, horizon=8):
    """Modèle ARIMA (VERSION DÉFINITIVE)"""
    print("\n" + "-"*50)
    print("📊 ARIMA")
    print("-"*50)
    try:
        # Convertir en numérique
        series = pd.to_numeric(series, errors='coerce').dropna()
        series = series.sort_index()
        
        if len(series) < 10:
            print(f"   ❌ Pas assez de données ({len(series)})")
            return None
        
        # CORRECTION: Définir une fréquence compatible avec statsmodels
        # Utiliser 'D' (daily) puis re-sampler sur les dates existantes
        # OU utiliser directement avec freq=None mais avec warning ignoré
        try:
            # Essayer d'inférer la fréquence
            freq = pd.infer_freq(series.index)
            if freq:
                series = series.asfreq(freq)
            else:
                # Si pas de fréquence détectée, utiliser un index numérique
                series = series.reset_index(drop=True)
        except:
            series = series.reset_index(drop=True)
        
        series = series.dropna()
        
        if len(series) < 10:
            print(f"   ❌ Pas assez de données ({len(series)})")
            return None
        
        # Test ADF
        try:
            result = adfuller(series)
            print(f"   Test ADF p-value: {result[1]:.4f}")
            if result[1] > 0.05:
                print("   ⚠️ Série non stationnaire")
            else:
                print("   ✅ Série stationnaire")
        except Exception as e:
            print(f"   ⚠️ Test ADF échoué: {e}")
        
        # Split
        train_size = int(len(series) * 0.8)
        train = series[:train_size]
        test = series[train_size:]
        
        print(f"   Train: {len(train)}, Test: {len(test)}")
        
        # Auto-ARIMA
        model_auto = auto_arima(
            train,
            start_p=0, start_q=0,
            max_p=3, max_q=3,
            seasonal=False,
            trace=False,
            error_action='ignore',
            suppress_warnings=True
        )
        
        # Entraînement
        model = ARIMA(train, order=model_auto.order)
        model_fit = model.fit()
        
        # Prédictions
        rmse, mae = None, None
        if len(test) > 1:
            try:
                pred = model_fit.forecast(steps=len(test))
                rmse = np.sqrt(mean_squared_error(test, pred))
                mae = mean_absolute_error(test, pred)
            except Exception as e:
                print(f"   ⚠️ Erreur prédiction: {e}")
        
        # Prévisions
        try:
            forecast = model_fit.forecast(steps=horizon)
        except:
            forecast = model_fit.predict(start=len(train), end=len(train)+horizon-1)
        
        print(f"   ✅ ARIMA{model_auto.order} | AIC: {model_fit.aic:.2f}")
        if rmse:
            print(f"   RMSE: {rmse:.2f}, MAE: {mae:.2f}")
        
        return {
            'model': model_fit, 'order': model_auto.order,
            'rmse': rmse, 'mae': mae, 'aic': model_fit.aic,
            'forecast': forecast, 'type': 'ARIMA'
        }
    except Exception as e:
        print(f"   ❌ Erreur ARIMA: {e}")
        return None

# Modèles SARIMA avec auto_arima pour la sélection des paramètres    
def modele_sarima(series, horizon=8, m=4):
    """Modèle SARIMA (VERSION DÉFINITIVE)"""
    print("\n" + "-"*50)
    print("📊 SARIMA")
    print("-"*50)
    try:
        # Convertir en numérique
        series = pd.to_numeric(series, errors='coerce').dropna()
        series = series.sort_index()
        
        if len(series) < 15:
            print(f"   ❌ Pas assez de données ({len(series)})")
            return None
        
        # CORRECTION: Utiliser index numérique
        try:
            freq = pd.infer_freq(series.index)
            if freq:
                series = series.asfreq(freq)
            else:
                series = series.reset_index(drop=True)
        except:
            series = series.reset_index(drop=True)
        
        series = series.dropna()
        
        train_size = int(len(series) * 0.8)
        train = series[:train_size]
        test = series[train_size:]
        
        # Auto-ARIMA saisonnier
        model_auto = auto_arima(
            train,
            start_p=0, start_q=0,
            max_p=2, max_q=2,
            seasonal=True, m=m,
            trace=False,
            error_action='ignore',
            suppress_warnings=True
        )
        
        # Entraînement
        model = SARIMAX(
            train,
            order=model_auto.order,
            seasonal_order=model_auto.seasonal_order
        )
        model_fit = model.fit(disp=False)
        
        # Prédictions
        rmse, mae = None, None
        if len(test) > 1:
            try:
                pred = model_fit.forecast(steps=len(test))
                rmse = np.sqrt(mean_squared_error(test, pred))
                mae = mean_absolute_error(test, pred)
            except Exception as e:
                print(f"   ⚠️ Erreur prédiction: {e}")
        
        # Prévisions
        try:
            forecast = model_fit.forecast(steps=horizon)
        except:
            forecast = model_fit.predict(start=len(train), end=len(train)+horizon-1)
        
        print(f"   ✅ SARIMA{model_auto.order}x{model_auto.seasonal_order}")
        print(f"   AIC: {model_fit.aic:.2f}")
        if rmse:
            print(f"   RMSE: {rmse:.2f}, MAE: {mae:.2f}")
        
        return {
            'model': model_fit, 'order': model_auto.order,
            'seasonal_order': model_auto.seasonal_order,
            'rmse': rmse, 'mae': mae, 'aic': model_fit.aic,
            'forecast': forecast, 'type': 'SARIMA'
        }
    except Exception as e:
        print(f"   ❌ Erreur SARIMA: {e}")
        return None

# Modèles SARIMAX avec auto_arima pour la sélection des paramètres
def modele_sarimax(series, exog_data, horizon=8, m=4):
    """
    Modèle SARIMAX (SARIMA + variables exogènes) - VERSION CORRIGÉE
    
    CORRECTIONS:
    ------------
    1. Gestion de l'index DatetimeIndex (comme ARIMA/SARIMA)
    2. Alignement correct des exogènes
    3. Gestion des erreurs de fréquence
    """
    print("\n" + "-"*50)
    print("📊 SARIMAX (avec variables exogènes)")
    print("-"*50)
    
    try:
        # ============================================================
        # ÉTAPE 1: Vérifications
        # ============================================================
        
        if len(series) < 15:
            print(f"   ❌ Pas assez de données ({len(series)})")
            return None
        
        if exog_data is None:
            print("   ⚠️ Pas d'exogènes fournies, utilisation de SARIMA standard")
            return modele_sarima(series, horizon, m)
        
        # ============================================================
        # ÉTAPE 2: CORRECTION - Gérer l'index de la série
        # ============================================================
        
        # Convertir en numérique
        series = pd.to_numeric(series, errors='coerce').dropna()
        series = series.sort_index()
        
        # Aligner les exogènes avec la série
        exog_data = exog_data.reindex(series.index).fillna(exog_data.mean())
        exog_data = exog_data.fillna(0)
        
        # ⭐ CORRECTION CLÉ : Gérer la fréquence comme ARIMA/SARIMA
        try:
            freq = pd.infer_freq(series.index)
            if freq:
                series_use = series.asfreq(freq)
                exog_use = exog_data.asfreq(freq)
            else:
                # Pas de fréquence détectée → index numérique
                series_use = series.reset_index(drop=True)
                exog_use = exog_data.reset_index(drop=True)
        except:
            series_use = series.reset_index(drop=True)
            exog_use = exog_data.reset_index(drop=True)
        
        # Supprimer les NaN
        series_use = series_use.dropna()
        exog_use = exog_use.loc[series_use.index] if len(series_use) == len(exog_use) else exog_use.iloc[:len(series_use)]
        
        if len(series_use) < 15:
            print(f"   ❌ Pas assez de données après nettoyage ({len(series_use)})")
            return None
        
        # ============================================================
        # ÉTAPE 3: Split train/test
        # ============================================================
        
        train_size = int(len(series_use) * 0.8)
        
        train = series_use[:train_size]
        test = series_use[train_size:]
        
        train_exog = exog_use[:train_size]
        test_exog = exog_use[train_size:]
        
        print(f"   📊 Train: {len(train)}, Test: {len(test)}")
        print(f"   📊 Exogènes: {exog_use.shape[1]} variables")
        
        # ============================================================
        # ÉTAPE 4: Auto-ARIMA avec exogènes
        # ============================================================
        
        print(f"   🔍 Recherche des paramètres...")
        
        model_auto = auto_arima(
            train,
            exogenous=train_exog,
            start_p=0, start_q=0,
            max_p=2, max_q=2,
            seasonal=True, m=m,
            trace=False,
            error_action='ignore',
            suppress_warnings=True,
            stepwise=True
        )
        
        print(f"   ✅ Paramètres: SARIMAX{model_auto.order}x{model_auto.seasonal_order}")
        
        # ============================================================
        # ÉTAPE 5: Entraîner SARIMAX
        # ============================================================
        
        model = SARIMAX(
            train,
            exog=train_exog,
            order=model_auto.order,
            seasonal_order=model_auto.seasonal_order
        )
        
        model_fit = model.fit(disp=False)
        
        # ============================================================
        # ÉTAPE 6: Prédictions sur le test
        # ============================================================
        
        rmse, mae = None, None
        
        if len(test) > 1:
            try:
                pred = model_fit.forecast(steps=len(test), exog=test_exog)
                rmse = np.sqrt(mean_squared_error(test, pred))
                mae = mean_absolute_error(test, pred)
                print(f"   ✅ Prédictions test OK")
            except Exception as e:
                print(f"   ⚠️ Erreur prédiction test: {e}")
        
        # ============================================================
        # ÉTAPE 7: Prévisions futures
        # ============================================================
        
        try:
            # Créer des exogènes futurs (moyenne des 4 dernières valeurs)
            exog_future = pd.DataFrame(
                np.tile(exog_use.iloc[-4:].mean().values, (horizon, 1)),
                columns=exog_use.columns
            )
            
            forecast = model_fit.forecast(steps=horizon, exog=exog_future)
            print(f"   ✅ Prévisions futures OK")
        except Exception as e:
            print(f"   ⚠️ Erreur prévision: {e}")
            # Fallback : prévisions simples
            forecast = np.full(horizon, series_use.iloc[-1])
        
        # ============================================================
        # RÉSULTAT
        # ============================================================
        
        print(f"   ✅ SARIMAX entraîné")
        print(f"   📊 AIC: {model_fit.aic:.2f}")
        if rmse:
            print(f"   📊 RMSE: {rmse:.2f}")
            print(f"   📊 MAE: {mae:.2f}")
        
        return {
            'model': model_fit,
            'order': model_auto.order,
            'seasonal_order': model_auto.seasonal_order,
            'exog_cols': exog_use.columns.tolist(),
            'rmse': rmse,
            'mae': mae,
            'aic': model_fit.aic,
            'forecast': forecast,
            'type': 'SARIMAX'
        }
        
    except Exception as e:
        print(f"   ❌ Erreur SARIMAX: {e}")
        import traceback
        traceback.print_exc()
        return None

# Modèles Holt-Winters avec ExponentialSmoothing pour la saisonnalité
def modele_holt_winters(series, horizon=8, sp=4):
    """Modèle Holt-Winters (VERSION DÉFINITIVE)"""
    print("\n" + "-"*50)
    print("📊 HOLT-WINTERS")
    print("-"*50)
    try:
        # Convertir en numérique
        series = pd.to_numeric(series, errors='coerce').dropna()
        series = series.sort_index()
        
        if len(series) < 8:
            print(f"   ❌ Pas assez de données ({len(series)})")
            return None
        
        # CORRECTION: Utiliser index numérique
        try:
            freq = pd.infer_freq(series.index)
            if freq:
                series = series.asfreq(freq)
            else:
                series = series.reset_index(drop=True)
        except:
            series = series.reset_index(drop=True)
        
        series = series.dropna()
        
        train_size = int(len(series) * 0.8)
        train = series[:train_size]
        test = series[train_size:]
        
        # Holt-Winters
        model = ExponentialSmoothing(
            train,
            trend='add',
            seasonal='add',
            seasonal_periods=sp,
            initialization_method='estimated'
        )
        model_fit = model.fit()
        
        # Prédictions
        rmse, mae = None, None
        if len(test) > 1:
            try:
                pred = model_fit.forecast(steps=len(test))
                rmse = np.sqrt(mean_squared_error(test, pred))
                mae = mean_absolute_error(test, pred)
            except Exception as e:
                print(f"   ⚠️ Erreur prédiction: {e}")
        
        # Prévisions
        try:
            forecast = model_fit.forecast(steps=horizon)
        except:
            forecast = np.full(horizon, series.iloc[-1])
        
        print(f"   ✅ Holt-Winters")
        if rmse:
            print(f"   RMSE: {rmse:.2f}, MAE: {mae:.2f}")
        
        return {
            'model': model_fit, 'rmse': rmse, 'mae': mae,
            'forecast': forecast, 'type': 'Holt-Winters'
        }
    except Exception as e:
        print(f"   ❌ Erreur Holt-Winters: {e}")
        return None
 
# Préparer les variables exogènes pour SARIMAX 
def preparer_exogenes(df, series, exog_cols=None, mode='auto'):
    """
    Prépare les variables exogènes pour SARIMAX - VERSION AMÉLIORÉE
    
    MODES:
    ------
    - 'auto'    : Sélection automatique par corrélation
    - 'top3'    : Top 3 exogènes
    - 'top5'    : Top 5 exogènes
    - 'all'     : Toutes les exogènes
    - liste     : Liste personnalisée
    
    ============================================================
    PRINCIPE DE SÉLECTION
    ============================================================
    
    On garde uniquement les exogènes qui ont :
    1. Une corrélation > 0.15 avec la cible
    2. Un écart-type significatif (variation dans le temps)
    3. Pas trop de NaN
    
    ============================================================
    """
    print("\n" + "-"*50)
    print("📊 PRÉPARATION DES VARIABLES EXOGÈNES")
    print("-"*50)
    
    # ============================================================
    # ÉTAPE 1: Identifier les exogènes candidates
    # ============================================================
    
    # Toutes les variables numériques sauf la cible
    var_num = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # Exclure les colonnes non pertinentes
    exclure = ['Nb_Cas', 'Num_IPD', 'Semaine_Num', 'Annee_Num']
    candidates = [v for v in var_num if v not in exclure]
    
    # ============================================================
    # ÉTAPE 2: Calculer les corrélations avec la cible
    # ============================================================
    
    print(f"\n   📊 Analyse des corrélations avec Nb_Cas:")
    print(f"   {'Variable':<25} {'Corrélation':>12} {'Verdict':<15}")
    print("   " + "-"*55)
    
    # Calculer la corrélation pour chaque candidate
    correlations = {}
    for col in candidates:
        try:
            # Corrélation entre la colonne et Nb_Cas
            corr = df[col].corr(df['Nb_Cas'])
            if pd.notna(corr):
                correlations[col] = abs(corr)
        except:
            pass
    
    # Trier par corrélation décroissante
    correlations = pd.Series(correlations).sort_values(ascending=False)
    
    # Afficher
    for col, corr in correlations.items():
        if corr > 0.3:
            verdict = "✅ FORTE"
        elif corr > 0.15:
            verdict = "⚠️ MOYENNE"
        else:
            verdict = "❌ FAIBLE"
        print(f"   {col:<25} {corr:>12.3f} {verdict}")
    
    # ============================================================
    # ÉTAPE 3: Sélection selon le mode
    # ============================================================
    
    if mode == 'auto':
        # Garder seulement les corrélations > 0.15
        selected = correlations[correlations > 0.15].index.tolist()
        print(f"\n   🎯 Mode AUTO: corrélation > 0.15")
        
    elif mode == 'top3':
        selected = correlations.head(3).index.tolist()
        print(f"\n   🎯 Mode TOP3: 3 meilleures exogènes")
        
    elif mode == 'top5':
        selected = correlations.head(5).index.tolist()
        print(f"\n   🎯 Mode TOP5: 5 meilleures exogènes")
        
    elif mode == 'all':
        selected = candidates
        print(f"\n   🎯 Mode ALL: toutes les exogènes")
        
    elif isinstance(mode, list):
        selected = [c for c in mode if c in df.columns]
        print(f"\n   🎯 Mode CUSTOM: {len(selected)} exogènes")
        
    else:
        # Par défaut : top 3
        selected = correlations.head(3).index.tolist()
        print(f"\n   🎯 Mode par défaut: TOP3")
    
    if not selected:
        print("   ⚠️ Aucune exogène sélectionnée")
        return None
    
    print(f"   ✅ {len(selected)} exogènes sélectionnées:")
    for i, col in enumerate(selected, 1):
        print(f"      {i}. {col} (corr: {correlations.get(col, 0):.3f})")
    
    # ============================================================
    # ÉTAPE 4: Agréger par Date
    # ============================================================
    
    if 'Date' not in df.columns:
        print("   ❌ Colonne 'Date' non trouvée")
        return None
    
    try:
        # Grouper par Date et prendre la moyenne
        df_exog = df.groupby('Date')[selected].mean()
        
        
        # ============================================================
        # ÉTAPE 5: Aligner avec la série temporelle
        # ============================================================
        
        df_exog = df_exog.reindex(series.index)
        df_exog = df_exog.fillna(df_exog.mean())
        df_exog = df_exog.fillna(0)
        
        print(f"\n   ✅ Exogènes préparées:")
        print(f"      - Shape: {df_exog.shape}")
        print(f"      - NA restants: {df_exog.isnull().sum().sum()}")
        
        # ============================================================
        # ÉTAPE 6: Vérifier la variation
        # ============================================================
        
        print(f"\n   📊 Statistiques:")
        print(f"   {'Variable':<20} {'Moyenne':>12} {'Écart-type':>12} {'Varie ?':<10}")
        print("   " + "-"*58)
        for col in df_exog.columns:
            mean = df_exog[col].mean()
            std = df_exog[col].std()
            var = "✅" if std > 0.01 * abs(mean) else "⚠️ Faible"
            print(f"   {col:<20} {mean:>12.2f} {std:>12.2f} {var:<10}")
        
       # ⭐ CORRECTION : Forcer float64
        print(f"\n   🔧 Types avant correction:")
        print(f"      {df_exog.dtypes.to_dict()}")
        
        df_exog = df_exog.astype(np.float64)
        
        print(f"   🔧 Types après correction:")
        print(f"      {df_exog.dtypes.to_dict()}") 
        
        
        return df_exog
        
    except Exception as e:
        print(f"   ❌ Erreur: {e}")
        return None
 
# Modèle Prophet (Facebook/Meta) pour la décomposition de la série temporelle 
def modele_prophet(series, horizon=8):
    """
    Modèle Prophet (Facebook/Meta) - VERSION CORRIGÉE
    
    CORRECTIONS:
    ------------
    1. Entraînement sur TOUTE la série avant prévisions futures
    2. Désactivation de la saisonnalité hebdomadaire (données hebdo)
    3. Activation de la saisonnalité mensuelle/annuelle
    4. Dates futures correctes
    """
    print("\n" + "-"*50)
    print("📊 PROPHET (Facebook/Meta)")
    print("-"*50)
    
    try:
        from prophet import Prophet
        import logging
        logging.getLogger('prophet').setLevel(logging.WARNING)
        logging.getLogger('cmdstanpy').setLevel(logging.WARNING)
        
        # ============================================================
        # ÉTAPE 1: Préparer les données
        # ============================================================
        df_prophet = pd.DataFrame({
            'ds': pd.to_datetime(series.index),
            'y': pd.to_numeric(series.values, errors='coerce')
        }).dropna()
        
        if len(df_prophet) < 10:
            print(f"   ❌ Pas assez de données ({len(df_prophet)})")
            return None
        
        print(f"   📊 Données: {len(df_prophet)} observations")
        
        # ============================================================
        # ÉTAPE 2: Split train/test POUR L'ÉVALUATION
        # ============================================================
        train_size = int(len(df_prophet) * 0.8)
        train = df_prophet[:train_size].copy()
        test = df_prophet[train_size:].copy()
        
        print(f"   📊 Train: {len(train)}, Test: {len(test)}")
        
        # ============================================================
        # ÉTAPE 3: Créer le modèle Prophet (CORRIGÉ)
        # ============================================================
        model = Prophet(
            growth='linear',
            changepoints=None,
            n_changepoints=10,
            
            # CORRECTION: Désactiver weekly, activer yearly
            yearly_seasonality='auto',      # ✅ Adapté aux données hebdo
            weekly_seasonality=False,       # ❌ Désactivé (données hebdo)
            daily_seasonality=False,
            
            # Saisonnalité mensuelle
            seasonality_mode='additive',
            seasonality_prior_scale=5.0,
            changepoint_prior_scale=0.05,
            
            interval_width=0.95,
            uncertainty_samples=1000
        )
        
        # Ajouter la saisonnalité mensuelle
        model.add_seasonality(
            name='monthly',
            period=30.5,
            fourier_order=5
        )
        
        print(f"   🔧 Configuration:")
        print(f"      - Tendance: {model.growth}")
        print(f"      - Saisonnalité: yearly + monthly")
        print(f"      - Mode: {model.seasonality_mode}")
        
        # ============================================================
        # ÉTAPE 4: Entraîner POUR L'ÉVALUATION (train uniquement)
        # ============================================================
        print(f"\n   🚀 Entraînement (train)...")
        model.fit(train)
        
        # ============================================================
        # ÉTAPE 5: Prédictions sur le test (ÉVALUATION)
        # ============================================================
        future_test = model.make_future_dataframe(periods=len(test), freq='W')
        forecast_test = model.predict(future_test)
        
        # Extraire les prédictions correspondant au test
        pred_test = forecast_test['yhat'].iloc[-len(test):].values
        
        # Métriques
        rmse, mae = None, None
        if len(test) > 1:
            rmse = np.sqrt(mean_squared_error(test['y'].values, pred_test))
            mae = mean_absolute_error(test['y'].values, pred_test)
        
        # ============================================================
        # ÉTAPE 6: RÉENTRAÎNER SUR TOUTE LA SÉRIE (CORRECTION)
        # ============================================================
        print(f"\n   🚀 Réentraînement sur toute la série...")
        
        # Créer un NOUVEAU modèle
        model_final = Prophet(
            growth='linear',
            n_changepoints=10,
            yearly_seasonality='auto',
            weekly_seasonality=False,
            daily_seasonality=False,
            seasonality_mode='additive',
            seasonality_prior_scale=5.0,
            changepoint_prior_scale=0.05,
            interval_width=0.95,
            uncertainty_samples=1000
        )
        
        model_final.add_seasonality(
            name='monthly',
            period=30.5,
            fourier_order=5
        )
        
        # Entraîner sur TOUTE la série
        model_final.fit(df_prophet)
        
        # ============================================================
        # ÉTAPE 7: Prévisions futures (CORRIGÉES)
        # ============================================================
        print(f"   🔮 Prévisions ({horizon} semaines)...")
        
        # Créer les dates futures
        future = model_final.make_future_dataframe(
            periods=horizon,
            freq='W',
            include_history=False
        )
        
        # Prédire
        forecast = model_final.predict(future)
        
        # Extraire
        forecast_values = forecast['yhat'].values
        forecast_lower = forecast['yhat_lower'].values
        forecast_upper = forecast['yhat_upper'].values
        forecast_dates = forecast['ds'].values
        
        print(f"\n   ✅ Prophet entraîné")
        if rmse:
            print(f"   📊 RMSE (évaluation): {rmse:.2f}")
            print(f"   📊 MAE (évaluation): {mae:.2f}")
        
        # Afficher les prévisions
        print(f"\n   📋 Prévisions Prophet (futures):")
        print("   " + "-"*60)
        print(f"   {'#':<4} {'Date':<15} {'Prévision':>12} {'IC Bas':>10} {'IC Haut':>10}")
        print("   " + "-"*60)
        for i in range(min(8, len(forecast_values))):
            date_str = pd.to_datetime(forecast_dates[i]).strftime('%d/%m/%Y')
            print(f"   {i+1:<4} {date_str:<15} {forecast_values[i]:>12.1f} "
                  f"{forecast_lower[i]:>10.1f} {forecast_upper[i]:>10.1f}")
        print("   " + "-"*60)
        
        return {
            'model': model_final,
            'rmse': rmse,
            'mae': mae,
            'aic': None,
            'forecast': forecast_values,
            'forecast_lower': forecast_lower,
            'forecast_upper': forecast_upper,
            'forecast_dates': forecast_dates,
            'type': 'Prophet'
        }
        
    except ImportError:
        print("   ❌ Prophet n'est pas installé")
        print("   💡 Installez avec: pip install prophet")
        return None
    except Exception as e:
        print(f"   ❌ Erreur Prophet: {e}")
        import traceback
        traceback.print_exc()
        return None
    
# ============================================================
# VISUALISATION PROPHET
# ============================================================

def visualiser_prophet(model, forecast_df):
    """
    Visualise les composantes de Prophet (VERSION CORRIGÉE)
    """
    print("\n   📊 Génération des visualisations Prophet...")
    
    if not os.path.exists('visualisations'):
        os.makedirs('visualisations')
    
    try:
        # ============================================================
        # GRAPHIQUE 1 : Prévisions Prophet (natif)
        # ============================================================
        fig = model.plot(forecast_df, figsize=(14, 6))
        plt.title('Prévisions Prophet - Cas Mpox Sénégal', 
                 fontsize=14, fontweight='bold')
        plt.xlabel('Date', fontsize=12)
        plt.ylabel('Nombre de cas', fontsize=12)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig('visualisations/prophet_previsions.png', 
                   dpi=300, bbox_inches='tight')
        print("      ✅ prophet_previsions.png")
        plt.close()
        
        # ============================================================
        # GRAPHIQUE 2 : Composantes Prophet (natif)
        # ============================================================
        fig2 = model.plot_components(forecast_df, figsize=(14, 8))
        fig2.suptitle('Composantes Prophet - Tendance et Saisonnalité',
                     fontsize=14, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.savefig('visualisations/prophet_composantes.png', 
                   dpi=300, bbox_inches='tight')
        print("      ✅ prophet_composantes.png")
        plt.close()
        
        # ============================================================
        # GRAPHIQUE 3 : Zoom personnalisé (CORRIGÉ)
        # ============================================================
        fig3, ax = plt.subplots(figsize=(14, 6))
        
        # Tracer l'intervalle de confiance
        ax.fill_between(
            forecast_df['ds'],
            forecast_df['yhat_lower'],
            forecast_df['yhat_upper'],
            color='#e74c3c',
            alpha=0.2,
            label='Intervalle de confiance 95%'
        )
        
        # Tracer les prévisions
        ax.plot(
            forecast_df['ds'],
            forecast_df['yhat'],
            color='#e74c3c',
            linewidth=2,
            label='Prévisions Prophet'
        )
        
        # Tracer les données historiques UNIQUEMENT si disponibles
        # ET de même longueur
        if 'y' in forecast_df.columns:
            # Filtrer les lignes où 'y' n'est pas NaN
            mask = forecast_df['y'].notna()
            if mask.sum() > 0:
                ax.scatter(
                    forecast_df.loc[mask, 'ds'],
                    forecast_df.loc[mask, 'y'],
                    color='#3498db',
                    s=30,
                    zorder=5,
                    label='Données observées'
                )
        
        ax.set_title('Prévisions Prophet avec intervalle de confiance',
                    fontsize=14, fontweight='bold')
        ax.set_xlabel('Date', fontsize=12)
        ax.set_ylabel('Nombre de cas', fontsize=12)
        ax.legend(loc='upper left')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig('visualisations/prophet_zoom.png', 
                   dpi=300, bbox_inches='tight')
        print("      ✅ prophet_zoom.png")
        plt.close()
        
        print("   ✅ Toutes les visualisations Prophet générées")
        
    except Exception as e:
        print(f"   ⚠️ Erreur visualisation Prophet: {e}")
        
# ============================================================
# FONCTION : Afficher les prévisions Prophet en tableau
# ============================================================

def afficher_previsions_prophet(resultats_prophet):
    """
    Affiche les prévisions Prophet sous forme de tableau
    """
    if resultats_prophet is None:
        return
    
    if 'forecast' not in resultats_prophet:
        return
    
    print("\n   📋 Prévisions Prophet:")
    print("   " + "-"*60)
    print(f"   {'#':<4} {'Date':<15} {'Prévision':>12} {'IC Bas':>10} {'IC Haut':>10}")
    print("   " + "-"*60)
    
    forecast = resultats_prophet['forecast']
    forecast_lower = resultats_prophet.get('forecast_lower', forecast * 0.8)
    forecast_upper = resultats_prophet.get('forecast_upper', forecast * 1.2)
    forecast_dates = resultats_prophet.get('forecast_dates', 
                                          pd.date_range(start=datetime.now(), periods=len(forecast), freq='W'))
    
    for i in range(min(8, len(forecast))):
        date = pd.to_datetime(forecast_dates[i]).strftime('%d/%m/%Y')
        val = forecast[i]
        lower = forecast_lower[i]
        upper = forecast_upper[i]
        
        print(f"   {i+1:<4} {date:<15} {val:>12.1f} {lower:>10.1f} {upper:>10.1f}")
    
    print("   " + "-"*60)
    
     
    
def comparer_modeles_ts(df, target='Nb_Cas', horizon=8):
    """Compare les modèles TS (VERSION CORRIGÉE)"""
    print("\n" + "="*70)
    print("COMPARAISON DES MODÈLES TS")
    print("="*70)
    
    if target not in df.columns:
        print(f"❌ Colonne '{target}' non trouvée")
        return None, {}, pd.DataFrame(), None
    
    if 'Date' not in df.columns:
        print("❌ Colonne 'Date' non trouvée")
        return None, {}, pd.DataFrame(), None
    
    # Diagnostic
    print(f"\n🔍 DIAGNOSTIC:")
    print(f"   - Type de 'Date': {df['Date'].dtype}")
    print(f"   - Dates non-NaT: {df['Date'].notna().sum()}/{len(df)}")
    
    # Créer la série temporelle
    df_clean = df.dropna(subset=['Date', target]).copy()
    df_clean['Date'] = pd.to_datetime(df_clean['Date'], errors='coerce')
    df_clean = df_clean.dropna(subset=['Date'])
    
    # Agréger par Date
    df_agg = df_clean.groupby('Date')[target].first().reset_index()
    df_agg = df_agg.sort_values('Date')
    
    # Créer la série SANS asfreq
    series = df_agg.set_index('Date')[target]
    series = pd.to_numeric(series, errors='coerce').dropna()
    series = series.sort_index()
    
    print(f"\n📊 Série temporelle:")
    print(f"   - {len(series)} observations")
    if len(series) > 0:
        print(f"   - Période: {series.index.min()} à {series.index.max()}")
        print(f"   - Moyenne: {series.mean():.2f}")
        print(f"   - Écart-type: {series.std():.2f}")
        print(f"   - Min: {series.min()}")
        print(f"   - Max: {series.max()}")
    
    if len(series) < 10:
        print(f"\n❌ Pas assez de données ({len(series)})")
        return None, {}, pd.DataFrame(), series
    
    results = {}
    
    # Arima
    r = modele_arima(series, horizon)
    if r: results['ARIMA'] = r
    
    # Sarima
    r = modele_sarima(series, horizon, m=4)
    if r: results['SARIMA'] = r
    
    # Holt-Winters
    r = modele_holt_winters(series, horizon, sp=4)
    if r: results['Holt-Winters'] = r
    
    # Prophet
    r = modele_prophet(series, horizon)
    if r: 
        results['Prophet'] = r
        
        # ⭐ Visualiser Prophet
        try:
            # Créer le forecast pour la visualisation (avec historique)
            future_viz = r['model'].make_future_dataframe(
                periods=horizon,
                freq='W',
                include_history=True  # ⭐ Inclure l'historique
            )
            forecast_viz = r['model'].predict(future_viz)
            
            # Ajouter les valeurs observées pour la visualisation
            df_obs = pd.DataFrame({
                'ds': series.index,
                'y': series.values
            })
            forecast_viz = forecast_viz.merge(df_obs, on='ds', how='left')
            
            # Appeler la fonction de visualisation
            visualiser_prophet(r['model'], forecast_viz)
            
        except Exception as e:
            print(f"   ⚠️ Erreur viz Prophet: {e}")
    
    
    # ⭐ SARIMAX - Tester 3 configurations
    print("\n" + "-"*50)
    print("📊 SARIMAX (3 configurations)")
    print("-"*50)
    
    # Configuration 1 : Top 3 exogènes (les meilleures)
    print("\n   🔧 Configuration 1: TOP 3 exogènes")
    exog_top3 = preparer_exogenes(df, series, mode='top3')
    if exog_top3 is not None:
        r = modele_sarimax(series, exog_top3, horizon, m=4)
        if r: 
            results['SARIMAX_TOP3'] = r
    
    # Configuration 2 : Top 5 exogènes
    print("\n   🔧 Configuration 2: TOP 5 exogènes")
    exog_top5 = preparer_exogenes(df, series, mode='top5')
    if exog_top5 is not None:
        r = modele_sarimax(series, exog_top5, horizon, m=4)
        if r: 
            results['SARIMAX_TOP5'] = r
    
    # Configuration 3 : Corrélation > 0.15
    print("\n   🔧 Configuration 3: Corrélation > 0.15")
    exog_auto = preparer_exogenes(df, series, mode='auto')
    if exog_auto is not None:
        r = modele_sarimax(series, exog_auto, horizon, m=4)
        if r: 
            results['SARIMAX_AUTO'] = r
    
    
    # Naïf
    print("\n" + "-"*50)
    print("📊 NAÏF")
    print("-"*50)
    try:
        train_size = int(len(series) * 0.8)
        train = series[:train_size]
        test = series[train_size:]
        
        if len(test) > 1:
            pred_naive = np.full(len(test), train.iloc[-1])
            rmse_naive = np.sqrt(mean_squared_error(test, pred_naive))
            mae_naive = mean_absolute_error(test, pred_naive)
            forecast_naive = np.full(horizon, series.iloc[-1])
            
            print(f"   ✅ RMSE: {rmse_naive:.2f}, MAE: {mae_naive:.2f}")
            
            results['Naïf'] = {
                'model': None, 'rmse': rmse_naive, 'mae': mae_naive,
                'forecast': forecast_naive, 'type': 'Naïf'
            }
    except Exception as e:
        print(f"   ❌ Erreur: {e}")
    
    # Moyenne mobile
    print("\n" + "-"*50)
    print("📊 MOYENNE MOBILE")
    print("-"*50)
    try:
        if len(series) > 4:
            train_size = int(len(series) * 0.8)
            train = series[:train_size]
            test = series[train_size:]
            
            if len(test) > 1:
                pred_ma = train.rolling(window=min(3, len(train))).mean().iloc[-1]
                pred_ma_array = np.full(len(test), pred_ma)
                rmse_ma = np.sqrt(mean_squared_error(test, pred_ma_array))
                mae_ma = mean_absolute_error(test, pred_ma_array)
                forecast_ma = np.full(horizon, pred_ma)
                
                print(f"   ✅ RMSE: {rmse_ma:.2f}, MAE: {mae_ma:.2f}")
                
                results['Moyenne Mobile'] = {
                    'model': None, 'rmse': rmse_ma, 'mae': mae_ma,
                    'forecast': forecast_ma, 'type': 'Moyenne Mobile'
                }
    except Exception as e:
        print(f"   ❌ Erreur: {e}")
    
    # Comparaison finale
    print("\n" + "="*70)
    print("📊 COMPARAISON FINALE")
    print("="*70)
    
    comparaison = pd.DataFrame(columns=['Modèle', 'RMSE', 'MAE', 'AIC'])
    for nom, res in results.items():
        comparaison.loc[len(comparaison)] = [
            nom, res.get('rmse', np.nan), res.get('mae', np.nan), res.get('aic', np.nan)
        ]
    
    if not comparaison.empty:
        print("\n" + comparaison.to_string(index=False))
        
        if 'RMSE' in comparaison.columns:
            meilleur_idx = comparaison['RMSE'].idxmin()
            meilleur = comparaison.loc[meilleur_idx]
            print(f"\n✅ MEILLEUR: {meilleur['Modèle']} (RMSE: {meilleur['RMSE']:.2f})")
        
        # Visualisation
        try:
            if not os.path.exists('visualisations'):
                os.makedirs('visualisations')
            
            fig, ax = plt.subplots(figsize=(12, 6))
            comp_clean = comparaison.dropna(subset=['RMSE'])
            if not comp_clean.empty:
                colors = ['blue', 'green', 'orange', 'red', 'purple'][:len(comp_clean)]
                bars = ax.bar(comp_clean['Modèle'], comp_clean['RMSE'], color=colors)
                ax.set_title('Comparaison RMSE des modèles TS', fontsize=14, fontweight='bold')
                ax.set_xlabel('Modèle')
                ax.set_ylabel('RMSE')
                ax.tick_params(axis='x', rotation=45)
                for bar, val in zip(bars, comp_clean['RMSE']):
                    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                           f'{val:.2f}', ha='center', va='bottom')
                plt.tight_layout()
                plt.savefig('visualisations/comparaison_modeles_ts.png', dpi=300, bbox_inches='tight')
                print("✅ visualisations/comparaison_modeles_ts.png")
                plt.close()
        except Exception as e:
            print(f"⚠️ Erreur viz: {e}")
        
        if 'RMSE' in comparaison.columns:
            meilleur_nom = comparaison.loc[comparaison['RMSE'].idxmin()]['Modèle']
            return results[meilleur_nom], results, comparaison, series
    
    return None, results, comparaison, series


#### 8- La fonction des visualisations

In [24]:
# ============================================================
# 8. VISUALISATIONS
# ============================================================

def visualisations_tendances(df, resultats_ts, series):
    """Visualisations des tendances"""
    print("\n" + "="*70)
    print("VISUALISATIONS DES TENDANCES")
    print("="*70)
    
    if not os.path.exists('visualisations'):
        os.makedirs('visualisations')
    
    # 1. Évolution avec tendance
    print("\n📊 1. Évolution avec tendance")
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(series.index, series.values, 'b-', linewidth=2, marker='o',
            markersize=4, label='Cas observés')
    
    if len(series) > 4:
        ma = series.rolling(window=4).mean()
        ax.plot(series.index, ma, 'orange', linewidth=2, linestyle='--',
                label='Moyenne mobile (4 sem.)')
    
    if len(series) > 2:
        x_numeric = np.arange(len(series))
        z = np.polyfit(x_numeric, series.values, 1)
        p = np.poly1d(z)
        ax.plot(series.index, p(x_numeric), 'red', linewidth=2, linestyle=':',
                label=f'Tendance (pente={z[0]:.2f})')
    
    ax.set_title('Évolution hebdomadaire avec tendance', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Nombre de cas')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('visualisations/tendance_evolution.png', dpi=300, bbox_inches='tight')
    print("   ✅ tendance_evolution.png")
    plt.close()
    
    # 2. Prévisions des modèles
    print("\n📊 2. Prévisions des modèles TS")
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(series.index, series.values, 'b-', linewidth=2, marker='o',
            markersize=4, label='Historique')
    
    colors = ['red', 'green', 'orange', 'purple', 'brown']
    for i, (nom, res) in enumerate(resultats_ts.items()):
        if 'forecast' in res and res['forecast'] is not None:
            forecast = res['forecast']
            if hasattr(forecast, 'values'):
                forecast = forecast.values
            
            last_date = series.index[-1]
            future_dates = pd.date_range(
                start=last_date + pd.Timedelta(days=7),
                periods=len(forecast), freq='W'
            )
            
            color = colors[i % len(colors)]
            ax.plot(future_dates, forecast, color=color, linewidth=2,
                    linestyle='--', marker='s', markersize=3,
                    label=f'{nom} (RMSE: {res.get("rmse", 0):.2f})')
    
    ax.set_title('Prévisions des cas par différents modèles', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Nombre de cas')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('visualisations/previsions_modeles.png', dpi=300, bbox_inches='tight')
    print("   ✅ previsions_modeles.png")
    plt.close()
    
    # 3. Saisonnalité
    if len(series) > 12:
        print("\n📊 3. Saisonnalité")
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        df_ts = pd.DataFrame({'Date': series.index, 'Nb_Cas': series.values})
        df_ts['Mois'] = df_ts['Date'].dt.month
        monthly = df_ts.groupby('Mois')['Nb_Cas'].mean()
        
        axes[0].bar(monthly.index, monthly.values, color='steelblue')
        axes[0].set_title('Moyenne des cas par mois', fontsize=12, fontweight='bold')
        axes[0].set_xlabel('Mois')
        axes[0].set_ylabel('Moyenne cas')
        axes[0].grid(True, alpha=0.3)
        
        df_ts['Trimestre'] = df_ts['Date'].dt.quarter
        trim = df_ts.groupby('Trimestre')['Nb_Cas'].mean()
        
        axes[1].bar(trim.index, trim.values, color='coral')
        axes[1].set_title('Moyenne des cas par trimestre', fontsize=12, fontweight='bold')
        axes[1].set_xlabel('Trimestre')
        axes[1].set_ylabel('Moyenne cas')
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig('visualisations/saisonnalite.png', dpi=300, bbox_inches='tight')
        print("   ✅ saisonnalite.png")
        plt.close()
    
    # 4. Décomposition
    print("\n📊 4. Décomposition")
    try:
        if len(series) >= 8:
            decomposition = seasonal_decompose(series, model='additive', period=4)
            fig, axes = plt.subplots(4, 1, figsize=(14, 10))
            
            axes[0].plot(decomposition.observed, 'b-')
            axes[0].set_title('Observé', fontweight='bold')
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(decomposition.trend, 'g-')
            axes[1].set_title('Tendance', fontweight='bold')
            axes[1].grid(True, alpha=0.3)
            
            axes[2].plot(decomposition.seasonal, 'orange')
            axes[2].set_title('Saisonnalité', fontweight='bold')
            axes[2].grid(True, alpha=0.3)
            
            axes[3].plot(decomposition.resid, 'red')
            axes[3].set_title('Résidus', fontweight='bold')
            axes[3].grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.savefig('visualisations/decomposition_serie.png', dpi=300, bbox_inches='tight')
            print("   ✅ decomposition_serie.png")
            plt.close()
    except Exception as e:
        print(f"   ⚠️ Erreur: {e}")

def visualisations_generales(df):
    """Visualisations générales"""
    print("\n" + "="*70)
    print("VISUALISATIONS GÉNÉRALES")
    print("="*70)
    
    if not os.path.exists('visualisations'):
        os.makedirs('visualisations')
    
    if 'Nom_region' in df.columns:
        print("\n📊 Distribution par région")
        fig, ax = plt.subplots(figsize=(12, 6))
        dist = df['Nom_region'].value_counts()
        dist.plot(kind='bar', ax=ax, color='darkgreen')
        ax.set_title('Distribution des cas par région', fontsize=14, fontweight='bold')
        ax.set_xlabel('Région')
        ax.set_ylabel('Nombre de cas')
        ax.tick_params(axis='x', rotation=45)
        for i, (region, count) in enumerate(dist.items()):
            ax.text(i, count + 1, str(count), ha='center', va='bottom')
        plt.tight_layout()
        plt.savefig('visualisations/distribution_regions.png', dpi=300, bbox_inches='tight')
        print("   ✅ distribution_regions.png")
        plt.close()
    
    if 'Age_Annee' in df.columns:
        print("\n📊 Distribution par âge")
        age_data = df['Age_Annee'].dropna()
        if not age_data.empty:
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            axes[0].hist(age_data, bins=20, color='blue', alpha=0.7, edgecolor='black')
            axes[0].axvline(age_data.mean(), color='red', linestyle='--',
                          label=f'Moyenne: {age_data.mean():.1f}')
            axes[0].set_title('Distribution de l\'âge', fontsize=12, fontweight='bold')
            axes[0].set_xlabel('Âge')
            axes[0].set_ylabel('Fréquence')
            axes[0].legend()
            axes[0].grid(True, alpha=0.3)
            
            axes[1].boxplot(age_data, vert=True, patch_artist=True)
            axes[1].set_title('Boxplot de l\'âge', fontsize=12, fontweight='bold')
            axes[1].set_ylabel('Âge')
            axes[1].grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.savefig('visualisations/dist_age.png', dpi=300, bbox_inches='tight')
            print("   ✅ dist_age.png")
            plt.close()
    
    if 'Sexe' in df.columns:
        print("\n📊 Distribution par sexe")
        sex_dist = df['Sexe'].value_counts()
        if not sex_dist.empty:
            fig, ax = plt.subplots(figsize=(8, 8))
            colors = ['#3498db', '#e74c3c', '#95a5a6'][:len(sex_dist)]
            sex_dist.plot(kind='pie', ax=ax, autopct='%1.1f%%',
                         colors=colors, explode=[0.05]*len(sex_dist))
            ax.set_title('Distribution par sexe', fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.savefig('visualisations/analyse_sexe.png', dpi=300, bbox_inches='tight')
            print("   ✅ analyse_sexe.png")
            plt.close()
    
    if 'Nom_region' in df.columns and 'Date' in df.columns:
        print("\n📊 Évolution top 5 régions")
        top_regions = df['Nom_region'].value_counts().head(5).index
        df_top = df[df['Nom_region'].isin(top_regions)]
        df_ts_top = df_top.groupby(['Date', 'Nom_region']).size().unstack(fill_value=0)
        
        if not df_ts_top.empty:
            fig, ax = plt.subplots(figsize=(14, 6))
            for region in top_regions:
                if region in df_ts_top.columns:
                    ax.plot(df_ts_top.index, df_ts_top[region], linewidth=2,
                           marker='o', markersize=3, label=region)
            ax.set_title('Évolution des 5 régions les plus touchées',
                        fontsize=14, fontweight='bold')
            ax.set_xlabel('Date')
            ax.set_ylabel('Nombre de cas')
            ax.legend(loc='upper left')
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig('visualisations/evolution_top_regions.png', dpi=300, bbox_inches='tight')
            print("   ✅ evolution_top_regions.png")
            plt.close()



#### 9- La fonction de génération de rapport final

In [25]:
# ============================================================
# 9. RAPPORT FINAL
# ============================================================

def generer_rapport_final(df, info_na, meilleur_modele_ts, resultats_ts, comparaison_ts, series):
    """Génère le rapport final"""
    print("\n" + "="*70)
    print("GÉNÉRATION DU RAPPORT FINAL")
    print("="*70)
    
    total_cas = len(df)
    regions = df['Nom_region'].nunique() if 'Nom_region' in df.columns else 0
    districts = df['Nom_District_Residence'].nunique() if 'Nom_District_Residence' in df.columns else 0
    
    info_serie = ""
    if series is not None and len(series) > 0:
        info_serie = f"""
Période: {series.index.min()} à {series.index.max()}
Nombre de semaines: {len(series)}
Moyenne hebdomadaire: {series.mean():.2f} cas
Écart-type: {series.std():.2f}
Min: {series.min():.0f} cas
Max: {series.max():.0f} cas
Dernière semaine: {series.iloc[-1]:.0f} cas"""
    
    comparaison_str = ""
    if comparaison_ts is not None and not comparaison_ts.empty:
        comparaison_str = comparaison_ts.to_string(index=False)
        if 'RMSE' in comparaison_ts.columns:
            meilleur = comparaison_ts.loc[comparaison_ts['RMSE'].idxmin()]
            comparaison_str += f"\n\n✅ MEILLEUR MODÈLE: {meilleur['Modèle']} (RMSE: {meilleur['RMSE']:.2f})"
    
    previsions_str = ""
    if meilleur_modele_ts and 'forecast' in meilleur_modele_ts:
        forecast = meilleur_modele_ts['forecast']
        if hasattr(forecast, 'values'):
            forecast = forecast.values
        
        last_date = series.index[-1]
        future_dates = pd.date_range(
            start=last_date + pd.Timedelta(days=7),
            periods=len(forecast), freq='W'
        )
        
        previsions_str = "\nPrévisions (8 semaines):\n"
        for i, (date, val) in enumerate(zip(future_dates, forecast)):
            if i < 8:
                previsions_str += f"   {date.strftime('%d/%m/%Y')}: {val:.1f} cas\n"
    
    na_str = f"""
Variables supprimées (>80% NA): {info_na['nb_supprimees']}
Variables imputées (simple): {info_na['nb_imputees_simple']}
Variables imputées (KNN): {info_na['nb_imputees_knn']}
"""
    
    rapport = f"""
============================================================
RAPPORT DE SURVEILLANCE MPOX - SÉNÉGAL
CIBLE : Nb_Cas (Nombre de cas par semaine)
Date: {datetime.now().strftime('%d/%m/%Y %H:%M')}
============================================================

1. SYNTHÈSE ÉPIDÉMIOLOGIQUE
---------------------------
Total des cas: {total_cas}
Régions touchées: {regions}
Districts touchés: {districts}

2. TRAITEMENT DES DONNÉES MANQUANTES
------------------------------------
{na_str}

3. TOP 5 RÉGIONS
----------------
"""
    
    if 'Nom_region' in df.columns:
        for region, count in df['Nom_region'].value_counts().head(5).items():
            rapport += f"    - {region}: {count} cas ({count/total_cas*100:.1f}%)\n"
    
    rapport += f"""
    
4. ANALYSE TEMPORELLE (Cible: Nb_Cas)
--------------------------------------
{info_serie}

5. COMPARAISON DES MODÈLES TS
-----------------------------
{comparaison_str}

6. PRÉVISIONS
-------------
{previsions_str}

7. RECOMMANDATIONS
------------------
"""
    
    if series is not None and len(series) > 4:
        recent_trend = series.tail(4).mean() - series.tail(8).head(4).mean()
        
        if recent_trend > 2:
            rapport += """
    🚨 ALERTE - Tendance à la hausse
    1. Renforcer la surveillance dans les régions touchées
    2. Activer les équipes d'intervention rapide
    3. Intensifier les tests de dépistage
"""
        elif recent_trend > 0:
            rapport += """
    ⚠️ SURVEILLANCE RENFORCÉE - Légère hausse
    1. Maintenir une veille épidémiologique active
    2. Vérifier la qualité des données collectées
"""
        else:
            rapport += """
    ✅ SITUATION MAÎTRISÉE - Tendance stable ou à la baisse
    1. Continuer la surveillance de routine
    2. Maintenir l'analyse hebdomadaire
"""
    
    rapport += f"""
    
============================================================
Rapport généré le {datetime.now().strftime('%d/%m/%Y à %H:%M')}
"""
    
    with open('rapport_final_mpox.txt', 'w', encoding='utf-8') as f:
        f.write(rapport)
    
    print(rapport)
    print("\n✅ Rapport sauvegardé: rapport_final_mpox.txt")
    return rapport


#### 10- La fonction Pipeline main regroupant toutes les fonctions

In [27]:
# ============================================================
# 10. FONCTION PRINCIPALE
# ============================================================

def main_complete():
    """Fonction principale complète - VERSION CORRIGÉE"""
    print("\n" + "="*70)
    print("SYSTÈME COMPLET DE SURVEILLANCE MPOX - SÉNÉGAL")
    print("CIBLE : Nb_Cas (Nombre de cas par semaine)")
    print("="*70)
    
    if not os.path.exists('visualisations'):
        os.makedirs('visualisations')
    
    # 1. Chargement
    print("\n📊 ÉTAPE 1: CHARGEMENT")
    base = charger_donnees()
    if base is None:
        return
    
    # 2. Préparation des données (crée Date ET Nb_Cas)
    print("\n📊 ÉTAPE 2: PRÉPARATION DES DONNÉES")
    df = preparer_donnees_complet(base)
    
    # Vérifier que Nb_Cas existe
    if 'Nb_Cas' not in df.columns:
        print("\n❌ ERREUR: La colonne 'Nb_Cas' n'a pas été créée !")
        print("   Colonnes disponibles:", df.columns.tolist()[:20])
        return
    
    print(f"\n✅ Cible Nb_Cas confirmée")
    
    # 3. Analyse NA (sur le df avec Nb_Cas)
    print("\n📊 ÉTAPE 3: ANALYSE DES NA")
    config_na = analyse_approfondie_na(df)
    
    # 4. Traitement NA
    print("\n📊 ÉTAPE 4: TRAITEMENT DES NA")
    df, info_na = traiter_na_avance(df, config_na)
    
    # Vérifier que Nb_Cas est toujours là
    if 'Nb_Cas' not in df.columns:
        print("\n❌ ERREUR: Nb_Cas a été supprimée lors du traitement NA !")
        return
    
    # 5. Sélection de variables
    print("\n📊 ÉTAPE 5: SÉLECTION DE VARIABLES")
    target = 'Nb_Cas'
    
    resultats_selection = selection_variables_avance(df, target, n_features=10)
    if resultats_selection:
        selected_features = resultats_selection['final_features']
        print(f"\n✅ {len(selected_features)} variables retenues")
    
    # 6. Modèles TS
    print("\n📊 ÉTAPE 6: MODÈLES TS")
    meilleur_modele_ts, resultats_ts, comparaison_ts, series = comparer_modeles_ts(
        df, target, horizon=8
    )
    
    # 7. Visualisations
    print("\n📊 ÉTAPE 7: VISUALISATIONS")
    visualisations_tendances(df, resultats_ts, series)
    visualisations_generales(df)
    
    # 8. Rapport
    print("\n📊 ÉTAPE 8: RAPPORT FINAL")
    generer_rapport_final(df, info_na, meilleur_modele_ts, resultats_ts, comparaison_ts, series)
    
    print("\n" + "="*70)
    print("✅ ANALYSE COMPLÈTE TERMINÉE")
    print("="*70)
    print("\n📁 Fichiers générés:")
    print("   - rapport_final_mpox.txt")
    print("   - visualisations/tendance_evolution.png")
    print("   - visualisations/previsions_modeles.png")
    print("   - visualisations/comparaison_modeles_ts.png")
    print("   - visualisations/saisonnalite.png")
    print("   - visualisations/decomposition_serie.png")
    print("   - visualisations/distribution_regions.png")
    print("   - visualisations/dist_age.png")
    print("   - visualisations/analyse_sexe.png")
    print("   - visualisations/evolution_top_regions.png")
# ============================================================
# EXÉCUTION
# ============================================================

if __name__ == "__main__":
    main_complete()


SYSTÈME COMPLET DE SURVEILLANCE MPOX - SÉNÉGAL
CIBLE : Nb_Cas (Nombre de cas par semaine)

📊 ÉTAPE 1: CHARGEMENT

CHARGEMENT DE LA BASE DE DONNÉES

RECHERCHE DU FICHIER DE DONNÉES

🔍 Recherche en cours...
✅ Fichier trouvé: Base_mpox_sen_S_05_2026.xlsx

✅ Base chargée avec succès
   - 474 lignes
   - 61 colonnes

📊 ÉTAPE 2: PRÉPARATION DES DONNÉES

PRÉPARATION DES DONNÉES

🔧 Traitement des dates

🔍 DIAGNOSTIC DATE:
   Type: datetime64[ns]
   Non-NaT: 474
   Exemples:
0   2024-08-12
1   2024-08-05
2   2024-08-19
3   2024-08-19
4   2024-08-12
   ✅ Dates créées pour 474 lignes

🎯 Création de Nb_Cas (par Date unique)
   ✅ Nb_Cas créée
   - Dates uniques: 61
   - Total cas: 8296
   - Moyenne: 7.77
   - Min/Max: 1/29
   ✅ Features temporelles créées

✅ Données préparées: 474 lignes, 70 colonnes

✅ Cible Nb_Cas confirmée

📊 ÉTAPE 3: ANALYSE DES NA

📊 ANALYSE APPROFONDIE DES VALEURS MANQUANTES

    📚 CAUSES POSSIBLES DES VALEURS MANQUANTES :
    --------------------------------------------
   